# Position-coded nodal RSAM heatmaps and robust array summaries

This notebook reads the RSAM archive recomputed from the position-coded SDS
archive. The station code is the along-line position in centimetres:

```text
station code 15010 -> 150.10 m
```

No serial-number mapping CSV or deployment-position lookup is required.
Because repeat surveys can encode the same nominal receiver position a few
centimetres differently, positions within a configurable tolerance are
clustered before plotting. Measurements from different deployment location
codes in the same position cluster are then merged by time and position.

The RSAM archive contains filenames ending in both `_59s.csv` and `_60s.csv`.
Both groups are read and combined.

Only robust amplitude measures are plotted:

- `median`: the one-minute median absolute amplitude;
- each bandpass metric:
  - `B4_8`
  - `B8_16`
  - `B16_32`
  - `B32_64`
  - `B64_128`
  - `B128_240`

For every available metric, network, and component, the notebook generates:

1. a position–time heatmap; and
2. an array-summary plot containing:
   - the median across receiver positions; and
   - a shaded 25th–75th percentile interquartile band.

Metrics such as `min`, `max`, `mean`, and `rms` are deliberately omitted
because they are more easily influenced by spikes, dropouts, clipping, or
other bad data.

Heatmap colour limits are shared across T1, T3, DPE, DPN, and DPZ separately
for each metric.

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from obspy import UTCDateTime

from flovopy.processing.sam import RSAM

## Configuration

In [ ]:
SAM_DIR = Path(
    "/Volumes/tachyon/LBSSP_DATA/nodal_rsam_position_codes"
)
OUTPUT_DIR = SAM_DIR / "all_metric_plots"

RUNS = {
    "T1": {
        "start": UTCDateTime("2026-05-16T16:00:00"),
        "end": UTCDateTime("2026-05-21T00:00:00"),
    },
    "T3": {
        "start": UTCDateTime("2026-05-19T00:00:00"),
        "end": UTCDateTime("2026-05-20T00:00:00"),
    },
}

CHANNELS = {
    "Z": "DPZ",
    "N": "DPN",
    "E": "DPE",
}

# FLOVOpy filenames in this archive use both nominal intervals.
SAMPLING_INTERVALS_S = [59, 60]
RSAM_EXTENSION = "csv"
METRICS = [
    "median",
    "B4_8",
    "B8_16",
    "B16_32",
    "B32_64",
    "B64_128",
    "B128_240",
]
CADENCE = "1min"

# Position codes from repeat deployments that differ by no more than this
# amount are treated as the same physical receiver location.
POSITION_TOLERANCE_M = 0.25

# Each merged station position is shown as a fixed-height band.
NODE_HEIGHT_M = 2.0
Y_RESOLUTION_M = 1.0

USE_LOG10 = True#False
GLOBAL_PERCENTILES = (2.0, 98.0)

HEATMAP_FIGSIZE = (13, 7)
AMPLITUDE_FIGSIZE = (13, 6)
DPI = 200

SUMMARY_LINE_WIDTH = 2.2
IQR_ALPHA = 0.25
VERBOSE_RSAM_READ = False

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SENSITIVITY = 2000 # V/m/s
print(f"RSAM root:  {SAM_DIR}")
print(f"Output dir: {OUTPUT_DIR}")

## Data preparation functions

In [ ]:
def parse_seed_id(seed_id: str) -> tuple[str, str, str, str]:
    """Split a SEED identifier into network, station, location, channel."""
    parts = seed_id.split(".")
    if len(parts) != 4:
        raise ValueError(
            f"Expected NET.STA.LOC.CHA, got {seed_id!r}"
        )
    return tuple(parts)


def station_code_to_position_m(station: str) -> float:
    """Convert a position-coded station name in centimetres to metres."""
    text = str(station).strip()

    if text.endswith(".0"):
        text = text[:-2]

    if not text.isdigit():
        raise ValueError(
            f"Station code {station!r} is not an integer centimetre position."
        )

    return int(text) / 100.0


def dataframe_time_series(
    dataframe: pd.DataFrame,
    metric: str,
) -> pd.Series:
    """Extract a UTC-indexed numeric RSAM metric series."""
    if metric not in dataframe.columns:
        raise KeyError(
            f"Metric {metric!r} not present; "
            f"available columns: {list(dataframe.columns)}"
        )

    if isinstance(dataframe.index, pd.DatetimeIndex):
        times = pd.to_datetime(dataframe.index, utc=True)
    else:
        time_column = next(
            (
                column
                for column in (
                    "time",
                    "datetime",
                    "date",
                    "timestamp",
                )
                if column in dataframe.columns
            ),
            None,
        )

        if time_column is None:
            numeric_index = pd.to_numeric(
                pd.Index(dataframe.index),
                errors="coerce",
            )

            if np.isfinite(numeric_index).all():
                times = pd.to_datetime(
                    numeric_index,
                    unit="s",
                    utc=True,
                )
            else:
                times = pd.to_datetime(
                    dataframe.index,
                    utc=True,
                    errors="coerce",
                )
        else:
            values = dataframe[time_column]
            times = pd.to_datetime(
                values,
                unit=(
                    "s"
                    if pd.api.types.is_numeric_dtype(values)
                    else None
                ),
                utc=True,
                errors="coerce",
            )

    values = pd.to_numeric(
        dataframe[metric],
        errors="coerce",
    ).to_numpy()

    series = pd.Series(values, index=times)
    return series[~series.index.isna()].sort_index()


def cluster_positions(
    positions: pd.Series,
    tolerance_m: float,
) -> tuple[pd.Series, pd.DataFrame]:
    """
    Merge nearby position codes that represent the same physical receiver.

    A cluster is built in sorted order and is allowed a total span no greater
    than ``tolerance_m``. Every member is replaced by the cluster mean.
    """
    if tolerance_m < 0:
        raise ValueError("tolerance_m cannot be negative.")

    unique_positions = np.sort(
        positions.dropna().unique().astype(float)
    )

    if len(unique_positions) == 0:
        return positions.copy(), pd.DataFrame()

    clusters: list[list[float]] = []
    current = [float(unique_positions[0])]

    for position in unique_positions[1:]:
        position = float(position)

        if position - current[0] <= tolerance_m + 1e-12:
            current.append(position)
        else:
            clusters.append(current)
            current = [position]

    clusters.append(current)

    position_lookup: dict[float, float] = {}
    summary_rows: list[dict[str, object]] = []

    for cluster_number, members in enumerate(clusters, start=1):
        mean_position = float(np.mean(members))

        for member in members:
            position_lookup[member] = mean_position

        summary_rows.append(
            {
                "cluster": cluster_number,
                "mean_position_m": mean_position,
                "minimum_position_m": min(members),
                "maximum_position_m": max(members),
                "span_m": max(members) - min(members),
                "original_position_count": len(members),
                "original_positions_m": ", ".join(
                    f"{member:.2f}" for member in members
                ),
            }
        )

    clustered = positions.map(
        lambda value: (
            position_lookup[float(value)]
            if pd.notna(value)
            else np.nan
        )
    )

    return clustered, pd.DataFrame(summary_rows)


def build_long_table(
    rsam: RSAM,
    *,
    metric: str,
    channel: str,
    position_tolerance_m: float,
) -> tuple[pd.DataFrame, pd.DataFrame, list[str]]:
    """Build a long table for one DP channel and merge nearby positions."""
    rows: list[pd.DataFrame] = []
    skipped: list[tuple[str, str]] = []
    matched_seed_ids: list[str] = []

    for seed_id, dataframe in rsam.dataframes.items():
        try:
            network, station, location, seed_channel = (
                parse_seed_id(seed_id)
            )

            if seed_channel != channel:
                continue

            original_position_m = station_code_to_position_m(
                station
            )
            series = dataframe_time_series(dataframe, metric)

        except Exception as exc:
            skipped.append((seed_id, str(exc)))
            continue

        values = series.to_numpy(dtype=float)
        good = np.isfinite(values)

        if not good.any():
            continue

        matched_seed_ids.append(seed_id)

        rows.append(
            pd.DataFrame(
                {
                    "time": series.index[good],
                    "value": values[good],
                    "network": network,
                    "station": station,
                    "location": location,
                    "channel": seed_channel,
                    "x_m_original": original_position_m,
                    "seed_id": seed_id,
                }
            )
        )

    if skipped:
        print(f"Skipped {len(skipped)} trace IDs. First examples:")
        for seed_id, reason in skipped[:15]:
            print(f"  {seed_id}: {reason}")

    if not rows:
        raise RuntimeError(
            f"No finite RSAM samples found for channel {channel!r}."
        )

    table = pd.concat(rows, ignore_index=True)

    table["x_m"], position_clusters = cluster_positions(
        table["x_m_original"],
        tolerance_m=position_tolerance_m,
    )

    return table, position_clusters, sorted(matched_seed_ids)

def read_rsam_multiple_intervals(
    *,
    start: UTCDateTime,
    end: UTCDateTime,
    sam_dir: Path,
    network: str,
    sampling_intervals_s: list[int],
    ext: str,
    verbose: bool = False,
) -> RSAM:
    """
    Read RSAM files written with more than one nominal sampling interval.

    FLOVOpy includes the inferred interval in each filename. In this archive,
    nominal one-minute products occur as both ``_59s`` and ``_60s`` files.
    Reading only one interval silently omits the other group.
    """
    combined = None
    loaded_by_interval: dict[int, int] = {}

    for interval_s in sampling_intervals_s:
        partial = RSAM.read(
            start,
            end,
            SAM_DIR=str(sam_dir),
            network=network,
            sampling_interval=interval_s,
            ext=ext,
            verbose=verbose,
        )

        n_loaded = len(getattr(partial, "dataframes", {}))
        loaded_by_interval[interval_s] = n_loaded

        if combined is None:
            combined = partial
        else:
            existing = getattr(combined, "dataframes", {})
            incoming = getattr(partial, "dataframes", {})

            duplicate_ids = sorted(set(existing) & set(incoming))
            if duplicate_ids:
                raise ValueError(
                    "The same RSAM IDs were returned for multiple nominal "
                    f"sampling intervals: {duplicate_ids[:10]}"
                )

            existing.update(incoming)

    if combined is None:
        raise RuntimeError("No sampling intervals were requested.")

    print(
        f"{network} files loaded by nominal interval: "
        + ", ".join(
            f"{interval_s}s={count}"
            for interval_s, count in loaded_by_interval.items()
        )
    )
    print(
        f"{network} combined RSAM dataframes: "
        f"{len(combined.dataframes)}"
    )

    return combined


## Grid and plotting functions

Rows from different deployment location codes are merged when they have the
same position-coded station name. If more than one value occurs at the same
position and minute, the median is used.

In [ ]:
def make_grid(
    table: pd.DataFrame,
    cadence: str,
) -> tuple[pd.DatetimeIndex, np.ndarray, np.ndarray]:
    """Create one resampled series per physical node position."""
    positions = np.sort(
        table["x_m"].dropna().unique().astype(float)
    )

    t0 = table["time"].min().floor(cadence)
    t1 = table["time"].max().ceil(cadence)

    times = pd.date_range(
        t0,
        t1,
        freq=cadence,
        tz="UTC",
    )

    grid = np.full(
        (len(positions), len(times)),
        np.nan,
    )

    for row_index, position_m in enumerate(positions):
        subset = table.loc[
            table["x_m"] == position_m,
            ["time", "value"],
        ]

        series = (
            subset
            .set_index("time")["value"]
            .sort_index()
            .resample(cadence)
            .median()
            .reindex(times)
        )

        grid[row_index, :] = series.to_numpy(dtype=float)

    return times, positions, grid



def grid_to_dataframe(
    times: pd.DatetimeIndex,
    positions: np.ndarray,
    grid: np.ndarray,
) -> pd.DataFrame:
    """Represent a position-by-time grid as a time-indexed DataFrame."""
    return pd.DataFrame(
        grid.T,
        index=times,
        columns=positions,
    )


def calculate_array_statistics(
    grid_dataframe: pd.DataFrame,
) -> pd.DataFrame:
    """Calculate robust spatial summaries at each time."""
    return pd.DataFrame(
        {
            "q25": grid_dataframe.quantile(
                0.25,
                axis=1,
                interpolation="linear",
            ),
            "median": grid_dataframe.median(
                axis=1,
                skipna=True,
            ),
            "q75": grid_dataframe.quantile(
                0.75,
                axis=1,
                interpolation="linear",
            ),
            "receiver_count": grid_dataframe.count(axis=1),
        },
        index=grid_dataframe.index,
    )
def make_fixed_height_position_grid(
    positions: np.ndarray,
    grid: np.ndarray,
    *,
    node_height_m: float,
    y_resolution_m: float,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Expand each node series into a fixed-height vertical band."""
    if node_height_m <= 0:
        raise ValueError("node_height_m must be positive.")

    if y_resolution_m <= 0:
        raise ValueError("y_resolution_m must be positive.")

    # Display at the nearest whole metre, as requested for the report figure.
    display_positions = np.rint(positions).astype(int)

    duplicate_counts = (
        pd.Series(display_positions)
        .value_counts()
        .loc[lambda values: values > 1]
    )

    if not duplicate_counts.empty:
        details = []

        for rounded_position in duplicate_counts.index:
            original_positions = positions[
                display_positions == rounded_position
            ]
            details.append(
                f"{rounded_position} m <- "
                + ", ".join(
                    f"{position:.2f} m"
                    for position in original_positions
                )
            )

        raise ValueError(
            "Rounding position-coded stations to whole metres merged "
            "distinct nodes:\n"
            + "\n".join(details)
        )

    half_height = node_height_m / 2.0

    y_min = np.floor(
        display_positions.min() - half_height
    )
    y_max = np.ceil(
        display_positions.max() + half_height
    )

    y_edges = np.arange(
        y_min,
        y_max + y_resolution_m,
        y_resolution_m,
        dtype=float,
    )

    y_lower = y_edges[:-1]
    y_upper = y_edges[1:]

    expanded_grid = np.full(
        (len(y_edges) - 1, grid.shape[1]),
        np.nan,
        dtype=float,
    )

    for source_row, display_position in enumerate(
        display_positions
    ):
        band_min = display_position - half_height
        band_max = display_position + half_height

        target_rows = (
            (y_lower >= band_min - 1e-9)
            & (y_upper <= band_max + 1e-9)
        )

        expanded_grid[target_rows, :] = grid[source_row, :]

    return display_positions, y_edges, expanded_grid


def datetime_bin_edges(
    times: pd.DatetimeIndex,
    cadence: str,
) -> np.ndarray:
    """Create explicit left-aligned pcolormesh time-bin edges."""
    if len(times) == 0:
        raise ValueError("No times supplied.")

    delta = pd.to_timedelta(cadence)

    edges = times.append(
        pd.DatetimeIndex([times[-1] + delta])
    )

    return mdates.date2num(edges.to_pydatetime())


def prepare_plot_data(
    table: pd.DataFrame,
    *,
    cadence: str,
    node_height_m: float,
    y_resolution_m: float,
    use_log10: bool,
) -> dict[str, object]:
    times, physical_positions, node_grid = make_grid(
        table,
        cadence,
    )
    # Convert from counts to ground velocity (m/s)
    node_grid = node_grid / SENSITIVITY

    grid_dataframe = grid_to_dataframe(
        times,
        physical_positions,
        node_grid,
    )
    array_statistics = calculate_array_statistics(
        grid_dataframe
    )

    display_positions, y_edges, display_grid = (
        make_fixed_height_position_grid(
            physical_positions,
            node_grid,
            node_height_m=node_height_m,
            y_resolution_m=y_resolution_m,
        )
    )

    if use_log10:
        with np.errstate(
            divide="ignore",
            invalid="ignore",
        ):
            heatmap_z = np.log10(display_grid)
    else:
        heatmap_z = display_grid.copy()

    heatmap_z[~np.isfinite(heatmap_z)] = np.nan

    if not np.isfinite(heatmap_z).any():
        raise RuntimeError(
            "No finite transformed values are available for plotting."
        )

    return {
        "times": times,
        "physical_positions": physical_positions,
        "display_positions": display_positions,
        "y_edges": y_edges,
        "node_grid": node_grid,
        "grid_dataframe": grid_dataframe,
        "array_statistics": array_statistics,
        "heatmap_z": heatmap_z,
    }
    
def plot_heatmap(
    plot_data: dict[str, object],
    *,
    outfile: Path,
    title: str,
    metric: str,
    channel: str,
    cadence: str,
    use_log10: bool,
    vmin: float,
    vmax: float,
    figsize: tuple[float, float],
    dpi: int,
) -> tuple[plt.Figure, plt.Axes]:
    times = plot_data["times"]
    display_positions = plot_data["display_positions"]
    y_edges = plot_data["y_edges"]
    z = plot_data["heatmap_z"]

    x_edges = datetime_bin_edges(times, cadence)

    fig, ax = plt.subplots(
        figsize=figsize,
        constrained_layout=True,
    )

    mesh = ax.pcolormesh(
        x_edges,
        y_edges,
        z,
        shading="flat",
        vmin=vmin,
        vmax=vmax,
    )

    ax.set_xlabel("Time (UTC)")
    ax.set_ylabel("Position along profile (m)")
    ax.set_ylim(y_edges[0], y_edges[-1])

    locator = mdates.AutoDateLocator(
        minticks=5,
        maxticks=9,
    )
    ax.xaxis.set_major_locator(locator)
    ax.xaxis.set_major_formatter(
        mdates.ConciseDateFormatter(locator)
    )

    if len(display_positions) <= 50:
        ax.set_yticks(display_positions)
        ax.tick_params(axis="y", labelsize=5)

    ax.grid(False)
    ax.set_title(title)

    colorbar = fig.colorbar(
        mesh,
        ax=ax,
        pad=0.015,
    )
    colorbar.set_label(
        (
            f"log10({metric} RSAM), {channel}"
            if use_log10
            else f"{metric} RSAM, {channel}"
        )
    )

    outfile.parent.mkdir(
        parents=True,
        exist_ok=True,
    )
    fig.savefig(
        outfile,
        dpi=dpi,
        bbox_inches="tight",
    )

    print(f"Wrote {outfile}")

    return fig, ax

def plot_heatmap(
    plot_data: dict[str, object],
    *,
    outfile: Path,
    title: str,
    metric: str,
    channel: str,
    cadence: str,
    use_log10: bool,
    vmin: float,
    vmax: float,
    figsize: tuple[float, float],
    dpi: int,
) -> tuple[plt.Figure, plt.Axes]:
    times = plot_data["times"]
    display_positions = plot_data["display_positions"]
    y_edges = plot_data["y_edges"]
    z = plot_data["heatmap_z"]

    x_edges = datetime_bin_edges(times, cadence)

    fig, ax = plt.subplots(
        figsize=figsize,
        constrained_layout=True,
    )

    mesh = ax.pcolormesh(
        x_edges,
        y_edges,
        z,
        shading="flat",
        vmin=vmin,
        vmax=vmax,
    )

    ax.set_xlabel("Time (UTC)")
    ax.set_ylabel("Position along profile (m)")
    ax.set_ylim(y_edges[0], y_edges[-1])

    locator = mdates.AutoDateLocator(
        minticks=5,
        maxticks=9,
    )
    ax.xaxis.set_major_locator(locator)
    ax.xaxis.set_major_formatter(
        mdates.ConciseDateFormatter(locator)
    )

    # Left axis: regular 10 m position ticks
    left_tick_min = 10 * np.ceil(y_edges[0] / 10)
    left_tick_max = 10 * np.floor(y_edges[-1] / 10)

    left_ticks = np.arange(
        left_tick_min,
        left_tick_max + 10,
        10,
    )

    ax.set_yticks(left_ticks)
    ax.tick_params(
        axis="y",
        labelsize=8,
    )

    # Right axis: exact displayed node positions
    ax_nodes = ax.twinx()
    ax_nodes.set_ylim(ax.get_ylim())
    ax_nodes.set_yticks(display_positions)
    ax_nodes.set_yticklabels(
        [f"{position:g}" for position in display_positions]
    )
    ax_nodes.tick_params(
        axis="y",
        labelsize=5,
        length=2,
        pad=2,
    )
    ax_nodes.set_ylabel("Node position (m)")

    ax.grid(False)
    ax_nodes.grid(False)
    ax.set_title(title)

    colorbar = fig.colorbar(
        mesh,
        ax=[ax, ax_nodes],
        pad=0.02,
    )
    colorbar.set_label(
        (
            f"log10({metric} RSAM), {channel}"
            if use_log10
            else f"{metric} RSAM, {channel}"
        )
    )

    outfile.parent.mkdir(
        parents=True,
        exist_ok=True,
    )
    fig.savefig(
        outfile,
        dpi=dpi,
        bbox_inches="tight",
    )

    print(f"Wrote {outfile}")

    return fig, ax

def plot_array_summary(
    plot_data: dict[str, object],
    *,
    outfile: Path,
    title: str,
    metric: str,
    channel: str,
    use_log10: bool,
    figsize: tuple[float, float],
    dpi: int,
    summary_line_width: float,
    iqr_alpha: float,
) -> tuple[plt.Figure, plt.Axes]:
    """
    Plot the spatial median with the 25th–75th percentile band.

    Spatial statistics are calculated in linear amplitude. Logarithms
    are applied only for display, so the line is log10(array median)
    and the band bounds are log10(array 25th and 75th percentiles).
    """
    statistics = plot_data["array_statistics"]

    q25 = statistics["q25"].to_numpy(dtype=float)
    median = statistics["median"].to_numpy(dtype=float)
    q75 = statistics["q75"].to_numpy(dtype=float)

    if use_log10:
        with np.errstate(
            divide="ignore",
            invalid="ignore",
        ):
            q25 = np.log10(q25)
            median = np.log10(median)
            q75 = np.log10(q75)

    q25[~np.isfinite(q25)] = np.nan
    median[~np.isfinite(median)] = np.nan
    q75[~np.isfinite(q75)] = np.nan

    fig, ax = plt.subplots(
        figsize=figsize,
        constrained_layout=True,
    )

    ax.fill_between(
        statistics.index,
        q25,
        q75,
        alpha=iqr_alpha,
        label="Interquartile range (25th–75th percentile)",
        zorder=1,
    )

    ax.plot(
        statistics.index,
        median,
        linewidth=summary_line_width,
        label="Median across positions",
        zorder=2,
    )
    
    
    if use_log10:
        finite = median[np.isfinite(median)]
        ymin, ymax = np.nanpercentile(finite, [0.5, 99.5])
        padding = 0.05 * (ymax - ymin)
        ax.set_ylim(ymin - padding, ymax + padding)

    ax.set_xlabel("Time (UTC)")
    ax.set_ylabel(
        (
            f"log10({metric} RSAM amplitude), {channel}"
            if use_log10
            else f"{metric} RSAM amplitude, {channel}"
        )
    )
    ax.set_title(title)

    locator = mdates.AutoDateLocator(
        minticks=5,
        maxticks=9,
    )
    ax.xaxis.set_major_locator(locator)
    ax.xaxis.set_major_formatter(
        mdates.ConciseDateFormatter(locator)
    )

    ax.legend()
    ax.grid(True, alpha=0.25)

    outfile.parent.mkdir(
        parents=True,
        exist_ok=True,
    )
    fig.savefig(
        outfile,
        dpi=dpi,
        bbox_inches="tight",
    )

    print(f"Wrote {outfile}")

    return fig, ax


## Load each network once

Each network is read once for `_59s` files and once for `_60s` files, then the returned dataframes are combined.


In [ ]:
rsam_by_network: dict[str, RSAM] = {}

for network, time_window in RUNS.items():
    print(
        f"Reading {network}: "
        f"{time_window['start']} to {time_window['end']}"
    )

    rsam = read_rsam_multiple_intervals(
        start=time_window["start"],
        end=time_window["end"],
        sam_dir=SAM_DIR,
        network=network,
        sampling_intervals_s=SAMPLING_INTERVALS_S,
        ext=RSAM_EXTENSION,
        verbose=VERBOSE_RSAM_READ,
    )

    rsam_by_network[network] = rsam

    print(
        f"  Loaded {len(rsam.dataframes)} RSAM dataframes."
    )

## Build every network/component/metric dataset

Each metric is prepared independently. Missing metrics or empty combinations
are reported and skipped without stopping the notebook.

In [ ]:
results: dict[
    tuple[str, str, str],
    dict[str, object],
] = {}

for network, rsam in rsam_by_network.items():
    for component, channel in CHANNELS.items():
        for metric in METRICS:
            key = (network, component, metric)

            print(
                f"\nPreparing {network} {channel} {metric}"
            )

            try:
                table, position_clusters, matched_seed_ids = (
                    build_long_table(
                        rsam,
                        metric=metric,
                        channel=channel,
                        position_tolerance_m=POSITION_TOLERANCE_M,
                    )
                )

                plot_data = prepare_plot_data(
                    table,
                    cadence=CADENCE,
                    node_height_m=NODE_HEIGHT_M,
                    y_resolution_m=Y_RESOLUTION_M,
                    use_log10=USE_LOG10,
                )

            except (
                RuntimeError,
                KeyError,
                ValueError,
            ) as exc:
                print(
                    f"  Skipping {network} {channel} "
                    f"{metric}: {exc}"
                )
                continue

            print(
                f"  Matched {len(matched_seed_ids)} SEED IDs"
            )
            print(
                f"  Original position codes: "
                f"{table['x_m_original'].nunique()}"
            )
            print(
                f"  Merged physical positions: "
                f"{table['x_m'].nunique()}"
            )
            print(
                f"  Location codes: "
                f"{sorted(table['location'].unique())}"
            )

            results[key] = {
                "metric": metric,
                "channel": channel,
                "table": table,
                "position_clusters": position_clusters,
                "matched_seed_ids": matched_seed_ids,
                "plot_data": plot_data,
            }

if not results:
    raise RuntimeError(
        "None of the requested RSAM datasets could be prepared."
    )

print(f"\nPrepared {len(results)} plot datasets.")

## Calculate shared heatmap colour limits for each metric

For each metric, finite transformed values are pooled across every successfully
prepared network and component. This gives one common heatmap scale per metric.

In [ ]:
metric_color_limits: dict[str, tuple[float, float]] = {}
metric_pool_sizes: dict[str, int] = {}

for metric in METRICS:
    metric_arrays = [
        result["plot_data"]["heatmap_z"][
            np.isfinite(
                result["plot_data"]["heatmap_z"]
            )
        ]
        for key, result in results.items()
        if key[2] == metric
    ]

    if not metric_arrays:
        print(
            f"No prepared datasets for metric {metric}; "
            "no colour limits calculated."
        )
        continue

    pooled_values = np.concatenate(metric_arrays)

    vmin, vmax = np.nanpercentile(
        pooled_values,
        GLOBAL_PERCENTILES,
    )

    metric_color_limits[metric] = (
        float(vmin),
        float(vmax),
    )
    metric_pool_sizes[metric] = int(
        pooled_values.size
    )

    print(
        f"{metric}: vmin={vmin:.6g}, "
        f"vmax={vmax:.6g}, "
        f"n={pooled_values.size:,}"
    )

In [ ]:
SURVEY_PERIODS = [
    {
        "label": "T1 Streamer",
        "start": "2026-05-16T17:06:00",
        "end": "2026-05-16T21:42:00",
    },
    {
        "label": "T2 Streamer",
        "start": "2026-05-17T13:54:00",
        "end": "2026-05-17T19:01:00",
    },
    {
        "label": "T1 1 m",
        "start": "2026-05-18T16:03:00",
        "end": "2026-05-18T18:39:00",
    },
    {
        "label": "T1 2 m",
        "start": "2026-05-18T20:18:00",
        "end": "2026-05-18T23:13:00",
    },
    {
        "label": "Nodal only",
        "start": "2026-05-19T12:59:00",
        "end": "2026-05-19T16:02:00",
    },
]

## Generate heatmaps and robust array-summary plots

Two files are written for every successfully prepared combination:

```text
<network>_<channel>_<metric>_heatmap.png
<network>_<channel>_<metric>_array_summary.png
```

The array-summary figure contains the spatial median and the
25th–75th percentile band.

In [ ]:
heatmap_figures = {}
summary_figures = {}
logstr = '_'
if USE_LOG10:
    logstr = 'log_'

for network in RUNS:
    for component, channel in CHANNELS.items():
        for metric in METRICS:
            key = (network, component, metric)

            if key not in results:
                continue

            if metric not in metric_color_limits:
                continue

            vmin, vmax = metric_color_limits[metric]

            metric_dir = OUTPUT_DIR / metric

            heatmap_outfile = (
                metric_dir
                / (
                    f"{network}_{channel}_{metric}_{logstr}"
                    "heatmap.png"
                )
            )
            summary_outfile = (
                metric_dir
                / (
                    f"{network}_{channel}_{metric}_{logstr}"
                    "array_summary.png"
                )
            )

            heatmap_title = (
                f"{network} nodal {channel} "
                f"{metric} amplitude through time"
            )
            summary_title = (
                f"{network} nodal {channel} "
                f"{metric} array median and interquartile range"
            )

            heatmap_fig, heatmap_ax = plot_heatmap(
                results[key]["plot_data"],
                outfile=heatmap_outfile,
                title=heatmap_title,
                metric=metric,
                channel=channel,
                cadence=CADENCE,
                use_log10=USE_LOG10,
                vmin=vmin,
                vmax=vmax,
                figsize=HEATMAP_FIGSIZE,
                dpi=DPI,
            )


            summary_fig, summary_ax = plot_array_summary(
                results[key]["plot_data"],
                outfile=summary_outfile,
                title=summary_title,
                metric=metric,
                channel=channel,
                use_log10=USE_LOG10,
                figsize=AMPLITUDE_FIGSIZE,
                dpi=DPI,
                summary_line_width=SUMMARY_LINE_WIDTH,
                iqr_alpha=IQR_ALPHA,
            )

            heatmap_figures[key] = (
                heatmap_fig,
                heatmap_ax,
            )
            summary_figures[key] = (
                summary_fig,
                summary_ax,
            )

            plt.show()
            plt.close(heatmap_fig)
            plt.close(summary_fig)

## Quality-control summary

In [ ]:
summary_rows = []

for (network, component, metric), result in results.items():
    table = result["table"]
    statistics = result["plot_data"]["array_statistics"]
    vmin, vmax = metric_color_limits.get(
        metric,
        (np.nan, np.nan),
    )

    summary_rows.append(
        {
            "network": network,
            "component": component,
            "channel": result["channel"],
            "metric": metric,
            "seed_ids": len(result["matched_seed_ids"]),
            "original_station_codes": (
                table["x_m_original"].nunique()
            ),
            "merged_station_positions": (
                table["x_m"].nunique()
            ),
            "location_codes": ", ".join(
                sorted(table["location"].unique())
            ),
            "minimum_position_m": table["x_m"].min(),
            "maximum_position_m": table["x_m"].max(),
            "samples": len(table),
            "maximum_receivers_per_time": int(
                statistics["receiver_count"].max()
            ),
            "heatmap_vmin": vmin,
            "heatmap_vmax": vmax,
        }
    )

summary = (
    pd.DataFrame(summary_rows)
    .sort_values(
        ["metric", "network", "component"]
    )
    .reset_index(drop=True)
)

summary

## Optional station-position inventory

This confirms that each station code converts cleanly to the expected
centimetre-based along-line position.

In [ ]:
# Position inventory is metric-independent, so use one prepared metric for
# each network/component combination.
inventory_tables = []
seen_network_components = set()

for (network, component, metric), result in results.items():
    network_component = (network, component)

    if network_component in seen_network_components:
        continue

    seen_network_components.add(network_component)

    inventory_tables.append(
        result["table"][
            [
                "network",
                "station",
                "location",
                "channel",
                "x_m_original",
                "x_m",
            ]
        ]
    )

position_inventory = (
    pd.concat(
        inventory_tables,
        ignore_index=True,
    )
    .drop_duplicates()
    .sort_values(
        [
            "network",
            "x_m",
            "location",
            "channel",
        ]
    )
    .reset_index(drop=True)
)

position_inventory

## Position-clustering quality control

Rows with more than one original position show repeated deployment codes
that were merged into one physical receiver location.

In [ ]:
cluster_qc = []
seen_network_components = set()

for (network, component, metric), result in results.items():
    network_component = (network, component)

    if network_component in seen_network_components:
        continue

    seen_network_components.add(network_component)

    clusters = result["position_clusters"].copy()
    clusters.insert(0, "component", component)
    clusters.insert(0, "network", network)
    cluster_qc.append(clusters)

cluster_qc = (
    pd.concat(cluster_qc, ignore_index=True)
    .sort_values(
        ["network", "component", "mean_position_m"]
    )
    .reset_index(drop=True)
)

cluster_qc.loc[
    cluster_qc["original_position_count"] > 1
]

## Expected output count

With 2 networks, 3 components, 7 robust metrics, and 2 plot types,
the maximum output is:

```text
2 × 3 × 7 × 2 = 84 PNG files
```

Missing network/component/metric combinations reduce that total.
Files are grouped into one output directory per metric.